In [2]:
import io, pickle
import numpy as np

dodaj train_data.zip a test_data.csv

# Trace Twins — BASELINE notebook

Run top to bottom. Replace the `Submission` below's `score_A`/`score_B` with your real methods**, then build and submit `submission.pkl`.

### 1. Setup (unzip the train data)

In [3]:
# The train data is in this notebook's folder as train_data.zip — unzip it.
!unzip -o train_data.zip          # -> public_traces.csv

Archive:  train_data.zip
  inflating: public_traces.csv       


In [4]:
import pandas as pd
df = pd.read_csv("public_traces.csv")
print(df.head())

   program_id category                                             tokens
0        5080   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
1        4607   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
2        2264   Adware  ntallocatevirtualmemory ntfreevirtualmemory nt...
3        3941   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...
4        4997   Adware  ldrloaddll ldrgetprocedureaddress ldrgetproced...


In [5]:
len(df)

4843

In [6]:
all_tokens = set()
for words in df["tokens"].str.split():
    all_tokens.update(words)
num_tokens = len(all_tokens)
print(f"Number of tokens: {num_tokens}")

Number of tokens: 275


In [7]:
token_labels = dict(zip(all_tokens, range(num_tokens)))

### 2. Your `Submission` (edit `score_A`/`score_B`)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [9]:
class SiameseNetwork(nn.Module):
    def __init__(self, num_tokens, d_model=16, nhead=1, num_layers=3):
        super(SiameseNetwork, self).__init__()
        self.embedding = nn.Embedding(num_tokens, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_tokens, d_model) )
        self.cls = nn.Parameter(torch.randn(1,1,d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, enable_nested_tensor=False)
    def forward(self, x):
        x = self.embedding(x)
        x = x + self.pos_embedding[:, :x.size(1), :]
        cls_token = self.cls.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.transformer_encoder(x)
        x = x[:, 0, :]
        return x

In [10]:
class Submission:
    def __init__(self):
        # Load or train your models here. The baseline currently needs nothing.
        self.model_A = SiameseNetwork(num_tokens,16,1,3)
        self.model_B = SiameseNetwork(num_tokens,16,1,3)

    @staticmethod
    def tokenize_A(window):
        pattern = []
        for word in window:
            pattern.append(token_labels[word])
        return torch.tensor(pattern)

    @staticmethod
    def tokenize_B(window):
        labels = {}
        pattern = []
        for word in window:
            if word not in labels:
                labels[word] = len(labels)
            pattern.append(labels[word])
        return torch.tensor(pattern)

    # ===== DO NOT EDIT: the cloud calls this to collect your scores =====
    def __call__(self, data: bytes) -> bytes:
        req = pickle.loads(data)
        fn = self.score_A if req["part"] == "A" else self.score_B
        scores = fn(req["windows"], req["pairs"])
        buf = io.BytesIO(); np.save(buf, np.asarray(list(scores), dtype=np.float64))
        return buf.getvalue()
    # ===================================================================

    def score_A(self, windows, pairs):
        tokenids = torch.stack([Submission.tokenize_A(window) for window in windows])
        embeddings = self.model_A(tokenids)
        pairs_t = torch.tensor(pairs)
        return [ max(0,score.item()) for score in F.cosine_similarity(embeddings[pairs_t[:,0]], embeddings[pairs_t[:,1]]) ]

    def score_B(self, windows, pairs):
        tokenids = torch.stack([Submission.tokenize_B(window) for window in windows])
        embeddings = self.model_B(tokenids)
        pairs_t = torch.tensor(pairs)
        return [ max(0,score.item()) for score in F.cosine_similarity(embeddings[pairs_t[:,0]], embeddings[pairs_t[:,1]]) ]

### 3. Train your solution (run once)

In [11]:
sol = Submission()   # loads/trains everything

In [12]:
def train(model, tokenizer, iters=10):
    model.to('cuda')
    model.train()
    criterion = nn.CosineEmbeddingLoss(margin=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    num = 20
    for it in range(iters):
        i = np.random.randint(len(df))
        j = np.random.randint(len(df))
        if i == j:
            continue
        row1 = df.iloc[i]
        row2 = df.iloc[j]
        id1 = row1["program_id"]
        id2 = row2["program_id"]
        if id1 == id2:
            continue
        tokens1 = row1["tokens"].split()
        tokens2 = row2["tokens"].split()
        if len(tokens1) < 200 or len(tokens2) < 200:
            continue

        idx1 = np.random.permutation(len(tokens1)-200+1)[:num]
        idx2 = np.random.permutation(len(tokens2)-200+1)[:num]

        token_ids1 = torch.stack([tokenizer(tokens1[idx:idx+200]) for idx in idx1])
        token_ids2 = torch.stack([tokenizer(tokens2[idx:idx+200]) for idx in idx2])

        x = torch.cat([token_ids1, token_ids2],dim=0)
        x = x.cuda()
        y = model(x)

        label = torch.tensor([id1]*len(idx1) + [id2]*len(idx2),dtype=torch.float32)
        label = label.cuda()

        y1 = y.repeat(y.shape[0], 1)
        y2 = y[:,None].repeat(1,y.shape[0],1).reshape(-1,y.shape[1])

        label1 = label.repeat(label.shape[0])
        label2 = label[:,None].repeat(1,label.shape[0]).reshape(-1)

        target = torch.where(label1 == label2, 1, -1)
        loss = criterion(y1, y2, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % 100 == 0:
            print(it, loss.item())

    model.eval()
    model.to('cpu')

In [13]:
train(sol.model_A,sol.tokenize_A,20000)

0 0.1327662467956543
100 0.16978789865970612
300 0.14496862888336182
400 0.04648233950138092
600 0.046337176114320755
700 0.15362536907196045
800 0.16625474393367767
900 0.14676061272621155
1200 0.2455318421125412
1300 0.09670808911323547
1400 0.054206881672143936
1500 0.048968005925416946
1600 0.08959876000881195
1800 0.12345771491527557
1900 0.1237187385559082
2000 0.041392307728528976
2100 0.2477293759584427
2200 0.113295778632164
2300 0.08460599929094315
2400 0.1970260590314865
2500 0.15586909651756287
2600 0.023778671398758888
2700 0.09012918919324875
2800 0.0713915228843689
2900 0.20522242784500122
3000 0.18149827420711517
3100 0.1269998401403427
3200 0.03385334089398384
3300 0.02791639417409897
3400 0.01586475409567356
3700 0.1230732724070549
3800 0.012799280695617199
3900 0.16257600486278534
4000 0.02460951916873455
4100 0.12833018600940704
4200 0.11101178079843521
4300 0.09759443253278732
4400 0.1261478066444397
4500 0.0775298923254013
4600 0.13373185694217682
4700 0.048373870

In [14]:
train(sol.model_B,sol.tokenize_B,30000)

0 0.2308446615934372
200 0.17952826619148254
300 0.07951591163873672
400 0.09941111505031586
500 0.11349509656429291
600 0.2429213672876358
700 0.2176218330860138
800 0.24194388091564178
900 0.030553894117474556
1000 0.17468780279159546
1100 0.23465682566165924
1200 0.03462499752640724
1300 0.1413460075855255
1400 0.2445734739303589
1500 0.24196891486644745
1600 0.028485922142863274
1700 0.17709881067276
1800 0.2502254247665405
1900 0.08500097692012787
2000 0.025562070310115814
2100 0.22102123498916626
2200 0.05441842973232269
2300 0.19254809617996216
2500 0.23060928285121918
2600 0.15484356880187988
2700 0.22644050419330597
2800 0.21342478692531586
2900 0.2256682962179184
3000 0.17645040154457092
3100 0.0712929219007492
3200 0.13141359388828278
3300 0.16293470561504364
3600 0.2206154614686966
3700 0.1754222959280014
3800 0.19744710624217987
3900 0.24209155142307281
4100 0.23563624918460846
4200 0.24304316937923431
4300 0.25025299191474915
4400 0.21546371281147003
4500 0.22691233456134

### 4. (optional) Estimate your score locally

In [15]:
# OPTIONAL local check — self-contained; mirrors the grader (disjoint Part A / Part B
# programs, per-window scramble for B, 50+50 bands). Needs only public_traces.csv.
import csv, random
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

WINDOW = 200; WPP = 8
def _load(p):
    out=[]
    with open(p) as f:
        r=csv.reader(f); next(r)
        for pid,cat,toks in r: out.append({"program_id":int(pid),"category":cat,"tokens":toks.split()})
    return out
def _windows(traces):
    out=[]
    for tr in traces:
        s=tr["tokens"]; n=(len(s)//WINDOW)*WINDOW
        out.extend([{"program_id":tr["program_id"],"category":tr["category"],"wid":j//WINDOW,
                     "tokens":s[j:j+WINDOW]} for j in range(0,n,WINDOW)][:WPP])
    return out
def _pairs(ws,n,seed):
    rng=random.Random(seed); bc=defaultdict(list); bp=defaultdict(list)
    for k,w in enumerate(ws): bc[w["category"]].append(k); bp[w["program_id"]].append(k)
    cats=sorted(bc); multi=[p for p in bp if len(bp[p])>=2]; P=[]; L=[]
    while len(P)<n:
        if rng.random()<0.5:
            p=rng.choice(multi); a,b=rng.sample(bp[p],2); P.append((a,b)); L.append(1)
        else:
            pool=bc[rng.choice(cats)]
            for _ in range(50):
                a,b=rng.sample(pool,2)
                if ws[a]["program_id"]!=ws[b]["program_id"]: P.append((a,b)); L.append(0); break
    return P,L
def _scramble(ws,off):
    vocab=sorted({t for w in ws for t in w["tokens"]}); out=[]
    for w in ws:
        r=random.Random((w["program_id"]*1_000_000+w["wid"])^off); sh=list(vocab); r.shuffle(sh)
        m=dict(zip(vocab,sh)); out.append([m[t] for t in w["tokens"]])
    return out

tr=_load("public_traces.csv")
by_prog={}
for t in tr: by_prog.setdefault(t["program_id"], t)
ids=sorted(by_prog); random.Random(7).shuffle(ids)
val=ids[:int(len(ids)*0.30)]; h=len(val)//2
wA=_windows([by_prog[i] for i in val[:h]]); WA=[w["tokens"] for w in wA]; pA,lA=_pairs(wA,3000,101)
wB=_windows([by_prog[i] for i in val[h:]]); WB=_scramble(wB,303); pB,lB=_pairs(wB,3000,202)
pts=lambda a,b: max(0.0, min(50.0,(a-0.5)/b*50.0))
aucA=roc_auc_score(lA, sol.score_A(WA,pA)); aucB=roc_auc_score(lB, sol.score_B(WB,pB))
print(f"Part A: AUC {aucA:.3f} -> {pts(aucA,0.34):.1f}/50")
print(f"Part B: AUC {aucB:.3f} -> {pts(aucB,0.28):.1f}/50")
print(f"ESTIMATED TOTAL ~ {pts(aucA,0.34)+pts(aucB,0.28):.1f}/100  (secret set differs slightly)")

Part A: AUC 0.789 -> 42.5/50
Part B: AUC 0.642 -> 25.4/50
ESTIMATED TOTAL ~ 67.8/100  (secret set differs slightly)


### 5. Build submission.pkl  (run LAST)

In [16]:
# Build submission.pkl  (this is what you upload as the Output)
import cloudpickle, os
# `sol` was trained above. Re-run the train cell first if you restarted the kernel.
with open("submission.pkl", "wb") as f:
    cloudpickle.dump(sol, f)
mb = os.path.getsize("submission.pkl") / 1e6
print(f"wrote submission.pkl  ({mb:.1f} MB)  -- must be < 50 MB")
assert mb < 50, "too big: cap model size (fewer trees / depth)"

wrote submission.pkl  (1.8 MB)  -- must be < 50 MB


In [17]:
mb

1.773724

### 6. Submit
Click the **🦆 Submit to Judge** button in the toolbar and choose `submission.pkl` as the Output.